# Matuszynska (2016) - NPQ Dynamics

[https://doi.org/10.1016/j.bbabio.2016.09.003](https://doi.org/10.1016/j.bbabio.2016.09.003)

In [ ]:
import math
from typing import Literal

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import gridspec
from matplotlib.patches import Rectangle
from mxlpy import Simulator, make_protocol, scan
from scipy.signal import peak_prominences

from mxlmodels import data, get_matuszynska2016_npq

# This model needs Assimulo

# Helper functions

In [ ]:
def cpfd(species: Literal["Arabidopsis", "Pothos"], PFD: float) -> float:
    """
    converts the given light intensity into internal activation rate
    calibrated for 3 light intensities for two species
    """
    if species == "Arabidopsis":
        light =  0.0005833 * PFD**2 + 0.2667 * PFD + 187.5
    elif species == "Pothos":
        light = 0.0004167 * PFD**2 + 0.3333 * PFD + 862.5

    return light

def phase_intervals_linear(
    phase_duration,
    step,
    first_interval = 0,
    num_steps = None,
    last_intervall = None
):
    phase = []
    accumulated_time = 0
    current_time = first_interval
    
    while accumulated_time < phase_duration and (num_steps is None or len(phase) < num_steps):
        accumulated_time += current_time
        if accumulated_time < phase_duration:
            phase.append(current_time)
        current_time = step * len(phase) + first_interval
        
    if last_intervall is not None:
        phase.append(phase_duration - last_intervall - sum(phase))
        if last_intervall != 0:
            phase.append(last_intervall)

    return phase

def npq(
    res_F,
    pam_prtc
):
    res_F_newindex = res_F.reset_index().rename(columns={"index": "time"}).copy()
    pulses = pam_prtc[pam_prtc["PPFD"] == pam_prtc["PPFD"].max()]
    pulses.index = pulses.index.total_seconds()
    res_extra = {
        "index": [],
        "Fm_time": [],
        "Fm": []
    }
    for i, time in enumerate(pulses.index):
        current_idx = (res_F_newindex["time"] - time).abs().idxmin()
        if i == 0:
            prior_idx = 0
        else:
            prior_idx = (res_F_newindex["time"] - pulses.index[i - 1]).abs().idxmin()
            prior_idx = current_idx - ((current_idx - prior_idx) // 2)
            
        if i == len(pulses.index) - 1:
            next_idx = len(res_F_newindex) - 1
        else:
            next_idx = (res_F_newindex["time"] - pulses.index[i + 1]).abs().idxmin()
            next_idx = current_idx + ((next_idx - current_idx) // 2)
            
        range_to_scan = res_F_newindex.iloc[prior_idx:next_idx]["Fluo"]

        res_extra["index"].append(range_to_scan.idxmax())
        res_extra["Fm_time"].append(res_F_newindex.loc[range_to_scan.idxmax(), "time"])
        res_extra["Fm"].append(range_to_scan.max())

    res_extra["NPQ"] = (res_extra["Fm"][0] - res_extra["Fm"]) / res_extra["Fm"]
    print(res_extra["index"])
    prominences, prominences_left, prominences_right = peak_prominences((res_F), res_extra["index"])
    
    res_extra["Fo"] = res_F.iloc[prominences_left].values
    res_extra["QY_PSII"] = (res_extra["Fm"] - res_extra["Fo"]) / res_extra["Fm"]
    
    return res_extra

# Figure 4

In [ ]:
phase_1 = [30]
phase_two = phase_intervals_linear(14*60, 20, 30)
phase_four = phase_intervals_linear(5*60, 20, 30, last_intervall=0)

sp_len = 0.8
sp_val = 5000

full_res = {}
full_prtc = {}

for pfd, phase_three_len in zip([100, 300, 900], [15, 30, 60]):
    phase_three = phase_intervals_linear(phase_three_len*60, 20, 30, num_steps=5, last_intervall=30)
    phase_prtc = []
    for phases, pfd_val in zip([phase_1, phase_two, phase_three, phase_four], [0, cpfd("Arabidopsis", pfd), 0, cpfd("Arabidopsis", pfd)], strict=False):
        for time in phases:
            phase_prtc.append((sp_len, {"PPFD": sp_val}))
            phase_prtc.append((time - sp_len, {"PPFD": pfd_val}))
    phase_prtc.append((5*60, {"PPFD": cpfd("Arabidopsis", pfd)}))
    
    pam_prtc = make_protocol(phase_prtc)
    s = Simulator(get_matuszynska2016_npq())
    s.simulate_protocol(pam_prtc, time_points_per_step=1000)
    full_res[pfd] = s.get_result().unwrap_or_err().get_combined()
    full_prtc[pfd] = pam_prtc

fig4_res, fig4_prtc = pd.concat(full_res, names=["PPFD"], axis=0), pd.concat(full_prtc, names=["PPFD"], axis=0)


In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(15, 3))

fig4_data = data.matuszynska2016_npq.default().fig4_data

box_height = 0.15
border_width = 0.01

for ax, pfd in zip(axs, fig4_res.index.levels[0]):
    this_res = fig4_res.loc[pfd]
    ax.plot(this_res["Fluo"] / max(this_res["Fluo"]), color="#db85d8", label="Simulated trace")
    
    pam_ptrc_peaks = fig4_prtc.loc[pfd][fig4_prtc.loc[pfd]["PPFD"] == 5000]
    peaks_times = pd.Series(pam_ptrc_peaks.index).apply(lambda x: x.total_seconds())
    ax.errorbar(peaks_times, fig4_data[pfd]["Fm_rel_mean"][:-1], yerr=fig4_data[pfd]["Fm_rel_std"][:-1], marker="^", color="black", linestyle="None", elinewidth=0.5, capsize=2.5, label=r"$\mathrm{F_M^'}$ measured", markersize=3.5)
    ax.errorbar(peaks_times, fig4_data[pfd]["Ft_rel_mean"][:-1], yerr=fig4_data[pfd]["Ft_rel_std"][:-1], marker="^", color="#0071ad", linestyle="None", elinewidth=0.5, capsize=2.5, label=r"$\mathrm{F_S}$ measured", markersize=3.5)
    
    xmin = -60
    if pfd == 100:
        ax.set_xlim(xmin, 36*60)
        ax.set_xticks([i*60 for i in [0, 14, 30, 35]], labels=[0, 14, 30, 35])
    elif pfd == 300:
        ax.set_xlim(xmin, 51*60)
        ax.set_xticks([i*60 for i in [0, 14, 45, 50]], labels=[0, 14, 45, 50])
    else:
        ax.set_xlim(xmin, 81*60)
        ax.set_xticks([i*60 for i in [0, 14, 75, 80]], labels=[0, 14, 75, 80])
    
    ax.set_ylim(0, 1.01)
    ax.set_yticks(np.linspace(0, 1, 6))
    
    cleaned_prtc = fig4_prtc.loc[pfd][fig4_prtc.loc[pfd]["PPFD"] != 5000]
    rect_coords = []
    for idx, val in cleaned_prtc.iterrows():
        if val["PPFD"] == 0:
            color="black"
        else:
            color="white"
        
        new_x = min(idx.total_seconds(), ax.get_xlim()[1])
        if rect_coords == []:
            rect_coords.append({"color": color, "start": xmin, "end": new_x})
        elif rect_coords[-1]["color"] == color:
            rect_coords[-1]["end"] = new_x
        else:
            prior_idx = rect_coords[-1]["end"]
            rect_coords.append({"color": color, "start": prior_idx, "end": new_x})
    
    for i, rect in enumerate(rect_coords):
        box = ax.add_patch(Rectangle(
            (rect["start"], ax.get_ylim()[-1]),
            width=abs(rect["end"] - rect["start"]),
            height=box_height,
            facecolor = rect["color"],
            edgecolor="black",
            clip_on=False
        ))
        
        if i == 1:
            xcoord = box.xy[0] + box.get_width() / 2
            ycoord = box.xy[1] + box.get_height() / 2
            text = f"{pfd} μE m$^{{-2}}$ s$^{{-1}}$" if pfd == 100 else f"{pfd}"
            ax.text(xcoord, ycoord, text, ha="center", va="center", weight="bold")
        
        if i == 2:
            xcoord = box.xy[0] + box.get_width() / 2
            ycoord = box.xy[1] + box.get_height() / 2
            ax.text(xcoord, ycoord, f"{math.ceil(box.get_width()/60)} min", ha="center", va="center", weight="bold", color="white")
            
    ax.set_xlabel("Time (min)")

axs[0].set(
    ylabel=r"Fluorescence ($\mathrm{F_M^'}$ / $\mathrm{F_M}$)",
)

handles, labels = axs[0].get_legend_handles_labels()

fig.legend(handles, labels, ncols=3, loc="lower center", shadow=True, framealpha=1, edgecolor="black", bbox_to_anchor=(0.5, 0.05))

plt.tight_layout()
plt.subplots_adjust(bottom=0.35)

# Figure 5

In [ ]:
time_changes = [30, 870, 1830, 2130]
time_steps = [time_changes[i] - time_changes[i-1] if i > 0 else time_changes[i] for i in range(len(time_changes))]

fig5_res = {
    "time_sim": {},
    "scan": {}
}
fig5_prtc = {}

for pfd in [100, 300, 900]:
    prtc = make_protocol([
        (time_steps[0], {"PPFD": 0}),
        (time_steps[1], {"PPFD": cpfd("Arabidopsis", pfd)}),
        (time_steps[2], {"PPFD": 0}),
        (time_steps[3], {"PPFD": cpfd("Arabidopsis", pfd)}),
    ])
    
    s = Simulator(get_matuszynska2016_npq())
    s.simulate_protocol(prtc, time_points_per_step=1000)
    fig5_res["time_sim"][pfd] = s.get_result().unwrap_or_err().get_combined()
    fig5_prtc[pfd] = prtc

steady_state = scan.steady_state(
    model=get_matuszynska2016_npq(),
    to_scan=pd.DataFrame({"PPFD": np.unique(np.sort(np.concatenate((np.linspace(0, 1000, 50), [100, 300, 900]))))})
)

fig5_res["scan"] = steady_state.combined

fig5_res["time_sim"] = pd.concat(fig5_res["time_sim"], names=["PPFD"], axis=0)
fig5_prtc = pd.concat(fig5_prtc, names=["PPFD"], axis=0)


In [ ]:
fig5 = plt.figure(figsize=(6, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1], hspace=0.3)
gs_top = gs[0].subgridspec(2, 1, hspace=0)

ax1 = fig5.add_subplot(gs_top[0, 0])
ax2 = fig5.add_subplot(gs_top[1, 0], sharex=ax1)
ax3 = fig5.add_subplot(gs[1, 0])

style_dict = {
    100: {"color": "#e09a05", "marker": "x"},
    300: {"color": "#06966b", "marker": "o"},
    900: {"color": "#1571a3", "marker": "s"},
}

for pfd in fig5_res["time_sim"].index.levels[0]:
    this_res = fig5_res["time_sim"].loc[pfd]
    this_res.index -= 30
    ax1.plot(this_res["pH_lu"], color=style_dict[pfd]["color"], label=str(pfd))
    ax2.plot(this_res["zx"], color=style_dict[pfd]["color"])
    ax2.plot(this_res["psbs_pr"], color=style_dict[pfd]["color"], linestyle="--")
    
    
    if pfd == 100:
        edgecolor = style_dict[pfd]["color"]
    else:
        edgecolor = "black"
    
    ax3.plot(this_res["Q"], this_res["pH_lu"], color=style_dict[pfd]["color"], marker=style_dict[pfd]["marker"], markerfacecolor=style_dict[pfd]["color"], markeredgecolor=edgecolor, markeredgewidth=0.5, label=str(pfd))

ax3.plot(fig5_res["scan"]["Q"], fig5_res["scan"]["pH_lu"], color="red")
ax3.scatter(fig5_res["scan"]["Q"].loc[np.array([100, 300, 900])], fig5_res["scan"]["pH_lu"].loc[np.array([100, 300, 900])], color="red", marker="o")

xmin = -20

ax1.set_xlim(xmin, 35*60)
xticks = fig5_prtc.loc[100].index.total_seconds() - 30
ax1.set_xticks(xticks, labels=xticks / 60)
ax2.set_xlabel("Time [min]")
plt.setp(ax1.get_xticklabels(), visible=False)

ax1.set_ylabel("lumen pH")
ax1.set_ylim(2, 8)
ax1.set_yticks(np.linspace(3, 8, 6))

ax2.set_ylabel("Q components")
ax2.set_ylim(0, 0.9)
ax2.set_yticks([0, 0.2, 0.5, 0.8])

ax3.set_xlabel("quencher activity [Q]")
ax3.set_xlim(0.05, 0.4)
ax3.set_xticks(np.linspace(0.1, 0.4, 4))
ax3.set_xticks(np.linspace(0.15, 0.35, 3),minor=True)

cleaned_prtc = fig5_prtc.loc[100]
rect_coords = []
for idx, val in cleaned_prtc.iterrows():
    color = "black" if val["PPFD"] == 0 else "white"
    
    new_x = min(idx.total_seconds() - 30, ax1.get_xlim()[1])
    if rect_coords == []:
        rect_coords.append({"color": color, "start": xmin, "end": new_x})
    elif rect_coords[-1]["color"] == color:
        rect_coords[-1]["end"] = new_x
    else:
        prior_idx = rect_coords[-1]["end"]
        rect_coords.append({"color": color, "start": prior_idx, "end": new_x})

for i, rect in enumerate(rect_coords):
    box = ax1.add_patch(Rectangle(
        (rect["start"], ax1.get_ylim()[-1]),
        width=abs(rect["end"] - rect["start"]),
        height=1,
        facecolor = rect["color"],
        edgecolor="black",
        clip_on=False
    ))

ax1.legend(loc="center left", bbox_to_anchor=(1.05, 0.5), frameon=True, edgecolor="black")
ax3.legend(loc="center left", bbox_to_anchor=(1.05, 0.5), frameon=True, edgecolor="black")

handles = []
for leg_head in ["PsbS$^P$", "Zx"]:
    handles.append(mlines.Line2D([], [], color="white", label=leg_head))
    for pfd in [100, 300, 900]:
        handles.append(mlines.Line2D([], [], color=style_dict[pfd]["color"], linestyle="--" if leg_head == "PsbS$^P$" else "-", label=str(pfd)))

ax2.legend(handles=handles, loc="center left", bbox_to_anchor=(1.05, 0.5), ncol=2, frameon=True, edgecolor="black")

plt.tight_layout()

# Figure 6


In [ ]:
fig6_m = get_matuszynska2016_npq()

fig6_m.update_parameter("gamma_2", 1)

phase_1 = [30]
phase_two = phase_intervals_linear(14*60, 20, 30)
phase_three = phase_intervals_linear(16*60, 20, 30, num_steps=5, last_intervall=30)
phase_four = phase_intervals_linear(5*60, 20, 30, last_intervall=0)

sp_len = 0.8
sp_val = 5000

phase_prtc = []
for phases, pfd_val in zip([phase_1, phase_two, phase_three, phase_four], [0, cpfd("Pothos", 100), 0, cpfd("Pothos", 100)], strict=False):
        for time in phases:
            phase_prtc.append((sp_len, {"PPFD": sp_val}))
            phase_prtc.append((time - sp_len, {"PPFD": pfd_val}))
phase_prtc.append((5*60, {"PPFD": cpfd("Pothos", 100)}))

pam_prtc = make_protocol(phase_prtc)
s = Simulator(fig6_m)
s.simulate_protocol(pam_prtc, time_points_per_step=1000)

fig6_res = s.get_result().unwrap_or_err().get_combined()
fig6_prtc = pam_prtc

In [ ]:
fig6_resextra = npq(
    res_F=fig6_res["Fluo"].copy(),
    pam_prtc=fig6_prtc
)

In [ ]:
fig6, axs = plt.subplots(nrows=2, figsize=(8, 6))

axs[0].plot(fig6_res["Fluo"] / fig6_res["Fluo"].max(), color="#fa0100", label="Simulated trace")
axs[1].plot(fig6_resextra["Fm_time"], fig6_resextra["QY_PSII"], color="#fa0100", marker="o", label="Simulation")

data_df = data.matuszynska2016_npq.default().fig6_data[100]
data_df.index = data_df["Timedelta_mean"]
axs[0].errorbar(fig6_resextra["Fm_time"], data_df["Fm_rel_mean"][:-1], yerr=data_df["Fm_rel_std"][:-1], marker="^", color="black", linestyle="None", elinewidth=0.5, capsize=2.5, label=r"$\mathrm{F_M^'}$ measured", markersize=3.5)
axs[0].errorbar(fig6_resextra["Fm_time"], data_df["Ft_rel_mean"][:-1], yerr=data_df["Ft_rel_std"][:-1], marker="^", color="#0071ad", linestyle="None", elinewidth=0.5, capsize=2.5, label=r"$\mathrm{F_S}$ measured", markersize=3.5)
axs[1].plot(data_df["Yield_mean"], marker="^", color="#0071ad", label=r"Experiment")

axs[0].set_ylabel("Fluorescence (a.u.)")
axs[0].set_ylim(0, 1)
axs[0].set_yticks(np.linspace(0, 1, 6))
axs[1].set_ylabel(r"$\Phi_{\mathrm{PSII}}$")
axs[1].set_ylim(0, 0.9)
axs[1].set_yticks(np.linspace(0, 0.8, 5))

cleaned_prtc = fig6_prtc[fig6_prtc["PPFD"] != 5000]
xmin = -2*60
for ax in axs:
    ax.set_xlim(xmin, 36*60)
    ax.set_xlabel("Time (min)")
    xticks_labels = [0, 14, 30, 35]
    ax.set_xticks([i*60 for i in xticks_labels], labels=xticks_labels)
    
    ax.legend(frameon=True, edgecolor="black", loc="center", bbox_to_anchor=(0.2, 0.75))
    
    box_height = (ax.get_ylim()[-1] - ax.get_ylim()[0]) * 0.15
    rect_coords = []
    for idx, val in cleaned_prtc.iterrows():
        color = "black" if val["PPFD"] == 0 else "white"
        
        new_x = min(idx.total_seconds(), ax.get_xlim()[1])
        if rect_coords == []:
            rect_coords.append({"color": color, "start": xmin, "end": new_x})
        elif rect_coords[-1]["color"] == color:
            rect_coords[-1]["end"] = new_x
        else:
            prior_idx = rect_coords[-1]["end"]
            rect_coords.append({"color": color, "start": prior_idx, "end": new_x})
    
    for i, rect in enumerate(rect_coords):
        box = ax.add_patch(Rectangle(
            (rect["start"], ax.get_ylim()[-1]),
            width=abs(rect["end"] - rect["start"]),
            height=box_height,
            facecolor = rect["color"],
            edgecolor="black",
            clip_on=False
        ))

plt.tight_layout()